In [ ]:
import os

from pyspark.sql import SparkSession

In [ ]:
JAR_PATH = "/opt/spark/notebooks/clickhouse-jdbc-0.6.5-shaded.jar"

spark = SparkSession.builder \
    .appName("explore_readmission_spark_app") \
    .config("spark.jars", JAR_PATH) \
    .config("spark.driver.extraClassPath", JAR_PATH) \
    .config("spark.executor.extraClassPath", JAR_PATH) \
    .getOrCreate()


spark.sparkContext.setLogLevel("ERROR")

In [ ]:
jdbc_url = f"jdbc:clickhouse://clickhouse:8123/{os.getenv('CLICKHOUSE_DB', 'default')}"

jdbc_options = {
    "url": jdbc_url,
    "user": os.getenv("CLICKHOUSE_USER", "default"),
    "password": os.getenv("CLICKHOUSE_PASSWORD", ""),
    "driver": "com.clickhouse.jdbc.ClickHouseDriver",
}

In [ ]:
query = """
SELECT
    patient_id,
    encounter_id,
    toString(start_at) AS start_at,
    toString(stop_at) AS stop_at,
    age,
    gender,
    race,
    marital_status,
    total_encounters_before,
    inpatient_count_before,
    emergency_count_before,
    avg_duration_before,
    total_conditions_before,
    chronic_conditions_before,
    distinct_conditions_codes_before,
    readmitted_30d
FROM ehr.features_readmission
"""


df = (
    spark.read.format("jdbc")
    .options(**jdbc_options)
    .option("dbtable", f"({query}) AS t")
    .load()
)

In [ ]:
df.printSchema()

In [ ]:
print(f"Rows: {df.count()}")
df.show(5)

In [ ]:
df.groupBy("readmitted_30d").count().show()

In [ ]:
numeric_columns = [
    "age", "total_encounters_before", "inpatient_count_before",
    "emergency_count_before", "avg_duration_before",
    "total_conditions_before", "chronic_conditions_before",
    "distinct_conditions_codes_before"
]

df.describe(numeric_columns).show()

In [ ]:
from pyspark.sql.functions import col, isnan, when

df.filter(isnan(col("avg_duration_before"))).count()

In [ ]:
df = df.withColumn(
    "avg_duration_before",
    when(isnan(col("avg_duration_before")) | col("avg_duration_before").isNull(), 0.0)
    .otherwise(col("avg_duration_before"))
)

In [ ]:
df.filter(isnan(col("avg_duration_before"))).count()

In [ ]:
df.describe("avg_duration_before").show()

In [ ]:
for numeric_col in numeric_columns:
    nan_count = df.filter(isnan(col(numeric_col))).count()
    null_count = df.filter(col(numeric_col).isNull()).count()
    print(f"{numeric_col}: NaN={nan_count}, NULL={null_count}")

In [ ]:
df.describe(numeric_columns).show()

In [ ]:
patient_ids = df.select(col("patient_id")).distinct()

train_ids, test_ids = patient_ids.randomSplit([0.6, 0.4], seed=42)

In [ ]:
train_df = df.join(train_ids, on="patient_id", how="inner")
test_df = df.join(test_ids, on="patient_id", how="inner")

In [ ]:
print(f"Rows (train): {train_df.count()}, rows (test): {test_df.count()}")

In [ ]:
train_df = train_df.withColumn(
    "gender_encoded",
    when(col("gender") == "M", 1).otherwise(0)
)
test_df = test_df.withColumn(
    "gender_encoded",
    when(col("gender") == "M", 1).otherwise(0)
)

In [ ]:
train_df.groupBy(col("gender"), col("gender_encoded")).count().show()

In [ ]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer

indexer = StringIndexer(inputCol="marital_status", outputCol="marital_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCol="marital_idx", outputCol="marital_onehot")

indexer_model = indexer.fit(train_df)
train_df = indexer_model.transform(train_df)
test_df = indexer_model.transform(test_df)

encoder_model = encoder.fit(train_df)
train_df = encoder_model.transform(train_df)
test_df = encoder_model.transform(test_df)

In [ ]:
train_df.select("gender", "gender_encoded", "marital_status", "marital_onehot").show(3, truncate=False)

In [ ]:
feature_cols = [
    "age",
    "total_encounters_before",
    "inpatient_count_before",
    "emergency_count_before",
    "avg_duration_before",
    "total_conditions_before",
    "chronic_conditions_before",
    "distinct_conditions_codes_before",
    "gender_encoded",
    "marital_onehot",
]
train_df.select(*feature_cols).show(3, truncate=False)

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_df = assembler.transform(train_df)
test_df = assembler.transform(test_df)

In [ ]:
print(f"Train rows: {train_df.count()}")
train_df.select("features", "readmitted_30d").show(3, truncate=False)

In [ ]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
scaler_model = scaler.fit(train_df)
train_df = scaler_model.transform(train_df)
test_df = scaler_model.transform(test_df)

In [ ]:
print(f"Train rows: {train_df.count()}")
train_df.select("features_scaled", "readmitted_30d").show(3, truncate=False)

In [ ]:
from pyspark.ml.classification import LogisticRegression

train_df = (
    train_df
        .withColumn("weight",
                    when(col("readmitted_30d") == 1, 5.0).otherwise(1.0))
)

test_df = (
    test_df
        .withColumn("weight",
                    when(col("readmitted_30d") == 1, 5.0).otherwise(1.0))
)

lr = LogisticRegression(
    featuresCol="features_scaled",
    labelCol="readmitted_30d",
    weightCol="weight",
    maxIter=100
)

lr_model = lr.fit(train_df)

In [ ]:
coefs = lr_model.coefficients.toArray()

names = [
    "age",
    "total_encounters_before",
    "inpatient_count_before",
    "emergency_count_before",
    "avg_duration_before",
    "total_conditions_before",
    "chronic_conditions_before",
    "distinct_conditions_codes_before",
    "gender_encoded",
    "marital_0",
    "marital_1",
    "marital_2",
    "marital_3",
]

for name, coef in sorted(zip(names, coefs), key=lambda x: abs(x[1]), reverse=True):
    print(f"{name:35s} {coef:+.4f}")

In [ ]:
predictions = lr_model.transform(test_df)
print(f"Predictions rows: {predictions.count()}")
predictions.show(3, truncate=False)

In [ ]:
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)

auc = BinaryClassificationEvaluator(
    labelCol="readmitted_30d",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
).evaluate(predictions)

print(f"AUC: {auc:.4f}")

recall_1 = MulticlassClassificationEvaluator(
    labelCol="readmitted_30d",
    predictionCol="prediction",
    metricName="recallByLabel",
    metricLabel=1.0
).evaluate(predictions)

print(f"Recall (class 1): {recall_1:.4f}")

precision_1 = MulticlassClassificationEvaluator(
    labelCol="readmitted_30d",
    predictionCol="prediction",
    metricName="precisionByLabel",
    metricLabel=1.0
).evaluate(predictions)

print(f"Precision (class 1): {precision_1:.4f}")

f1 = MulticlassClassificationEvaluator(
    labelCol="readmitted_30d",
    predictionCol="prediction",
    metricName="f1"
).evaluate(predictions)

print(f"F1: {f1:.4f}")

In [ ]:
print("Confusion Matrix")
predictions.groupBy("readmitted_30d", "prediction").count().orderBy("readmitted_30d", "prediction").show()

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType

extract_prob_udf = udf(lambda v: float(v[1]) if v is not None else None, FloatType())

predictions_to_save = predictions.select(
    "patient_id",
    "encounter_id",
    extract_prob_udf(col("probability")).alias("risk_score"),
    col("prediction").cast("int").alias("prediction"),
    col("readmitted_30d").cast("int").alias("actual")
)

In [ ]:
predictions_to_save.show(3)

In [ ]:
(
    predictions_to_save.write
    .format("jdbc")
    .options(**jdbc_options)
    .option("dbtable", "ehr.mart_predictions")
    .mode("append")
    .option("truncate", "true")
    .save()
)